# Multi-Task Learning: PANAS Presence (Task 1) & Likert-Intensitaet (Task 2)

**Gemeinsamer Encoder + drei Heads (Presence, Likert, Cosine-Auxiliary), synchron trainiert (Joint Training).**

## Datenrollen

- `data4training_syn_deep_seek.csv` wird ausschliesslich fuer Train/Validation/Test, Ablation und Checkpoint-Auswahl genutzt.
- Synthetische Rohwerte: `0 = nicht enthalten`, `1-5 = Likert-Intensitaet`; daraus werden Presence-Labels und Likert-Masken gebildet.
- `realdata.csv` wird ausschliesslich als externe Likert-Validierung verwendet; es fliesst nicht in Training, Test oder Checkpoint-Auswahl ein.
- Da der echte Fragebogen alle Items erzwingt, gibt es fuer Realdata keine Presence-Ground-Truth. Presence-Metriken werden dort nicht berechnet; die Coverage zeigt nur, fuer wie viele gueltige Ratings das Modell einen Presence-Score oberhalb des Schwellwerts liefert.
- Reale Likert-Metriken werden nur fuer gueltige Ratings ausgewertet, deren Item der Presence-Head aktiviert. Das ist eine **bedingte** Likert-Auswertung und keine vollstaendige End-to-End-Bewertung.

## Ablation: mit vs. ohne Cosine-Loss

Abschnitt 5 trainiert dieselbe Architektur mehrfach mit `LAMBDA_COSINE = 0.0` (kein Cosine-Loss) und
`LAMBDA_COSINE = 0.5` (mit Cosine-Loss), auf identischem synthetischem Train/Validation/Test-Split. Anschliessend werden beide Checkpoints unabhaengig auf `realdata.csv` anhand der bedingten Likert-Vorhersagen verglichen.

Der Checkpoint wird nach `L_presence + lambda * L_likert` auf dem **Validierungssplit** gewaehlt (**ohne** Cosine-Anteil), damit beide Varianten mit demselben Kriterium verglichen werden. Der synthetische Testsplit wird erst am ausgewaehlten Checkpoint berichtet.

In [2]:
import gc
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from functools import partial
from IPython.display import display
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, mean_absolute_error, mean_squared_error, cohen_kappa_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

c:\Users\StrasserNora\Desktop\NorasNotizen\privat\NLE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ---------------------------------------------------------------------------
# Reproduzierbare Pfad-, Daten- und Trainingskonfiguration
# ---------------------------------------------------------------------------
# Die CSV-Dateien liegen neben diesem Notebook.
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data4training_syn_deep_seek.csv"
REAL_DATA_PATH = BASE_DIR / "realdata.csv"
CSV_SEP = ";"

# Kommentar 1: Der synthetische Text steht in Antwort_Text.
TEXT_COLUMN = "Antwort_Text"
PANAS_ITEMS = [
    "aktiv", "bekuemmert", "interessiert", "freudig_erregt", "veraergert",
    "stark", "schuldig", "erschrocken", "feindselig", "angeregt",
    "stolz", "gereizt", "begeistert", "beschaemt", "wach", "nervoes",
    "entschlossen", "aufmerksam", "durcheinander", "aengstlich",
]
NUM_ITEMS = len(PANAS_ITEMS)

# Kommentar 2: Die Realdata nutzt dieselben Items mit deutscher Schreibweise.
REAL_TEXT_COLUMN = "audio"
REAL_GROUP_COLUMN = "name"
REAL_PANAS_COLUMNS = [
    "aktiv", "bekümmert", "interessiert", "freudig erregt", "verärgert",
    "stark", "schuldig", "erschrocken", "feindselig", "angeregt",
    "stolz", "gereizt", "begeistert", "beschämt", "wach", "nervös",
    "entschlossen", "aufmerksam", "durcheinander", "ängstlich",
]

# Kommentar 3: Mehrere Schwellenwerte zeigen den Trade-off zwischen Fehler und Coverage.
REAL_PRESENCE_THRESHOLDS = [0.3, 0.5, 0.7, 0.9]
REAL_PRESENCE_THRESHOLD = 0.5

# Kommentar 4: Die neue Gewichtung bleibt für alle Encoder identisch.
POS_WEIGHT_MAX = 20.0
LAMBDA_PRESENCE = 3.0

# Kommentar 5: BERT und DistilBERT werden unter denselben Trainingsbedingungen verglichen.
ENCODER_CONFIGS = {
    "bert": {"model_name": "bert-base-german-cased", "label": "BERT"},
    "distilbert": {"model_name": "dbmdz/distilbert-base-german-cased", "label": "DistilBERT"},
}
MODEL_NAME = ENCODER_CONFIGS["bert"]["model_name"]
SIM_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
TEST_SIZE = 0.2
RANDOM_STATE = 42
SEEDS = [42, 43, 44]
EXPERIMENTS = {
    "bert_ohne_cosine": {"encoder": "bert", "lambda_cosine": 0.0},
    "bert_mit_cosine": {"encoder": "bert", "lambda_cosine": 0.5},
    "distilbert_ohne_cosine": {"encoder": "distilbert", "lambda_cosine": 0.0},
    "distilbert_mit_cosine": {"encoder": "distilbert", "lambda_cosine": 0.5},
}

# L_total = lambda_presence * L_presence + lambda_cosine * L_cosine + lambda_likert * L_likert
LAMBDA_LIKERT = 1.0
LAMBDA_COSINE = 0.5
LIKERT_LOSS_TYPE = "mse"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for required_path in (DATA_PATH, REAL_DATA_PATH):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Datendatei nicht gefunden: {required_path}. "
            "Bitte das Notebook aus dem Ordner mit den CSV-Dateien starten."
        )

print(f"Device: {device}")
print(f"Trainingsdaten: {DATA_PATH}")
print(f"Externe Validierungsdaten: {REAL_DATA_PATH}")
print(f"PANAS-Items: {NUM_ITEMS}")
print("Encoder-Vergleich:", ", ".join(config["label"] for config in ENCODER_CONFIGS.values()))

Device: cpu
Trainingsdaten: c:\Users\StrasserNora\Desktop\NorasNotizen\privat\NLE\data4training_syn_deep_seek.csv
Externe Validierungsdaten: c:\Users\StrasserNora\Desktop\NorasNotizen\privat\NLE\realdata.csv
PANAS-Items: 20
Encoder-Vergleich: BERT, DistilBERT


### Methodische Entscheidung: Datenrollen und Splits

Die synthetischen Daten werden für Training, Validierung und den internen Test verwendet, während die Realdata ausschließlich als externe Validierung dient. Dadurch wird vermieden, dass Informationen aus den echten Daten in das Training oder die Checkpoint-Auswahl gelangen.

Der zusätzliche Validierungssplit ist notwendig, weil der Testsplit nicht zur Auswahl des besten Checkpoints verwendet werden sollte. Andernfalls würde die Testleistung indirekt optimiert und wäre zu positiv eingeschätzt. Mehrere Seeds reduzieren außerdem das Risiko, dass der Vergleich zwischen den Cosine-Loss-Varianten von einer einzelnen Zufallsinitialisierung abhängt.

## 1. Data Pipeline

In [4]:
df_synth = pd.read_csv(DATA_PATH, sep=CSV_SEP, dtype=str)

text_all = df_synth[TEXT_COLUMN].fillna("").to_numpy()

# Rohwerte 0 (nicht enthalten) bis 5 (Likert-Intensität) -> gleichzeitig Regressionsziel
# für Task 2 UND Grundlage für die Presence-Ableitung von Task 1.
raw_targets = df_synth[PANAS_ITEMS].apply(pd.to_numeric, errors="coerce")
target_values_all = raw_targets.fillna(0).to_numpy(dtype=np.float32)

# Task 1 (Presence/Gating): 0 = nicht enthalten, 1 = enthalten (Item hat einen Wert 1-5).
# Deine 0/1-Logik direkt aus den Rohwerten abgeleitet -- kein separates codiertes File nötig.
target_mask_all = (target_values_all > 0).astype(np.float32)

print(f"Anzahl Samples: {len(text_all)}")
print(f"Presence-Anteil pro Item (Mittelwert über alle Items): {target_mask_all.mean():.3f}")


Anzahl Samples: 1400
Presence-Anteil pro Item (Mittelwert über alle Items): 0.071


In [5]:

sim_model = SentenceTransformer(SIM_MODEL_NAME)
item_embeddings = sim_model.encode(PANAS_ITEMS, normalize_embeddings=True)

def cosine_targets(transcript):
    sentences = [part.strip() for part in str(transcript).split(".") if part.strip()]
    if not sentences:
        return np.zeros(NUM_ITEMS, dtype=np.float32)
    sentence_embeddings = sim_model.encode(sentences, normalize_embeddings=True)
    return util.cos_sim(sentence_embeddings, item_embeddings).cpu().numpy().max(axis=0)

cosine_values_all = np.vstack([cosine_targets(t) for t in text_all]).astype(np.float32)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1763.15it/s]


In [6]:
# Train/Validation/Test-Split der synthetischen Daten -- ALLE Arrays synchron.
# Das Testset bleibt unangetastet für die abschließende interne Bewertung.
(text_train_val, text_test,
 target_values_train_val, target_values_test,
 cosine_train_val, cosine_test,
 target_mask_train_val, target_mask_test) = train_test_split(
    text_all, target_values_all, cosine_values_all, target_mask_all,
    test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
(text_train, text_val,
 target_values_train, target_values_val,
 cosine_train, cosine_val,
 target_mask_train, target_mask_val) = train_test_split(
    text_train_val, target_values_train_val, cosine_train_val, target_mask_train_val,
    test_size=TEST_SIZE, random_state=RANDOM_STATE,
)

print(f"Synthetisch: {len(text_train)} Train-Samples | {len(text_val)} Validierungs-Samples | {len(text_test)} Test-Samples")

Synthetisch: 896 Train-Samples | 224 Validierungs-Samples | 280 Test-Samples


In [7]:
# ---------------------------------------------------------------------------
# Dataset & DataLoader (synthetisches Train + Validation + Test), dynamisches Padding
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class PanasMultiTaskDataset(Dataset):
    """Haelt Rohtexte + Targets; Tokenisierung passiert im collate_fn (dynamic padding)."""

    def __init__(self, texts, target_values, target_mask, cosine_values):
        self.texts = texts
        self.target_values = target_values
        self.target_mask = target_mask
        self.cosine_values = cosine_values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": str(self.texts[idx]),
            "target_values": self.target_values[idx],
            "target_mask": self.target_mask[idx],
            "cosine_values": self.cosine_values[idx],
        }


def collate_fn(batch, tokenizer, max_length):
    texts = [item["text"] for item in batch]
    encodings = tokenizer(
        texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
    )
    target_values = torch.tensor(
        np.stack([item["target_values"] for item in batch]), dtype=torch.float32
    )
    target_mask = torch.tensor(
        np.stack([item["target_mask"] for item in batch]), dtype=torch.float32
    )
    cosine_values = torch.tensor(
        np.stack([item["cosine_values"] for item in batch]), dtype=torch.float32
    )
    return {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "target_values": target_values,
        "target_mask": target_mask,
        "cosine_values": cosine_values,
    }


train_dataset = PanasMultiTaskDataset(text_train, target_values_train, target_mask_train, cosine_train)
val_dataset = PanasMultiTaskDataset(text_val, target_values_val, target_mask_val, cosine_val)
test_dataset = PanasMultiTaskDataset(text_test, target_values_test, target_mask_test, cosine_test)

collate = partial(collate_fn, tokenizer=tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

print(f"Synthetische Train-Batches: {len(train_loader)} | Val-Batches: {len(val_loader)} | Test-Batches: {len(test_loader)}")

Synthetische Train-Batches: 112 | Val-Batches: 28 | Test-Batches: 35


In [8]:
# Externe Validierung auf echten Likert-Antworten; nicht zum Training/Splitten verwenden.
real_df = pd.read_csv(REAL_DATA_PATH, sep=CSV_SEP, dtype=str)
missing_real_columns = set([REAL_TEXT_COLUMN, REAL_GROUP_COLUMN, *REAL_PANAS_COLUMNS]).difference(real_df.columns)
if missing_real_columns:
    raise ValueError(f"Fehlende Spalten in Realdata: {sorted(missing_real_columns)}")

real_df = real_df.loc[real_df[REAL_TEXT_COLUMN].fillna("").str.strip().ne("")].reset_index(drop=True)
real_text = real_df[REAL_TEXT_COLUMN].fillna("").to_numpy()
real_groups = real_df[REAL_GROUP_COLUMN].fillna("<missing_name>").to_numpy()
real_raw_targets = real_df[REAL_PANAS_COLUMNS].apply(pd.to_numeric, errors="coerce")
real_target_mask = (
    real_raw_targets.notna() & real_raw_targets.ge(1) & real_raw_targets.le(5)
).to_numpy(dtype=np.float32)
real_target_values = real_raw_targets.fillna(0).to_numpy(dtype=np.float32)
real_cosine_values = np.zeros_like(real_target_values, dtype=np.float32)  # Nicht für Realdata-Metriken verwendet.
real_dataset = PanasMultiTaskDataset(
    real_text, real_target_values, real_target_mask, real_cosine_values
)
real_collate = partial(collate_fn, tokenizer=tokenizer, max_length=MAX_LENGTH)
real_loader = DataLoader(
    real_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=real_collate
)
print(f"Externe Realdata-Validierung: {len(real_dataset)} Transkripte")
print(f"Eindeutige Personen/Name-Gruppen: {pd.Series(real_groups).nunique()}")
print(f"Gültige Likert-Ratings: {int(real_target_mask.sum())}")

# Datenqualitäts- und Truncation-Check: Diese Warnungen bitte vor der Interpretation prüfen.
invalid_real_values = real_raw_targets.notna() & ~(real_raw_targets.ge(1) & real_raw_targets.le(5))
invalid_count = int(invalid_real_values.to_numpy().sum())
missing_count = int(real_raw_targets.isna().to_numpy().sum())
real_token_lengths = [
    len(tokenizer(str(text), add_special_tokens=True, truncation=False)["input_ids"])
    for text in real_text
]
truncated_count = sum(length > MAX_LENGTH for length in real_token_lengths)
print("\n" + "!" * 78)
print("DATENQUALITÄT / TRUNCATION-CHECK")
print(f"Ungültige Likert-Werte außerhalb 1..5: {invalid_count}")
print(f"Fehlende Likert-Werte: {missing_count}")
print(f"Transkripte länger als MAX_LENGTH={MAX_LENGTH}: {truncated_count}/{len(real_text)}")
if truncated_count:
    print("WARNUNG: Diese Transkripte werden für das Modell abgeschnitten. MAX_LENGTH=512 als Sensitivitätsanalyse prüfen.")
else:
    print("OK: Kein Realdata-Transkript überschreitet MAX_LENGTH.")
print("!" * 78)

Externe Realdata-Validierung: 120 Transkripte
Eindeutige Personen/Name-Gruppen: 36
Gültige Likert-Ratings: 2180

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
DATENQUALITÄT / TRUNCATION-CHECK
Ungültige Likert-Werte außerhalb 1..5: 0
Fehlende Likert-Werte: 220
Transkripte länger als MAX_LENGTH=256: 11/120
WARNUNG: Diese Transkripte werden für das Modell abgeschnitten. MAX_LENGTH=512 als Sensitivitätsanalyse prüfen.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


### Methodische Entscheidung: Externe Validierung

Die Realdata enthält echte Audio-Transkripte und Likert-Werte, aber keine unabhängige Ground Truth dafür, ob ein PANAS-Item im gesprochenen Text tatsächlich genannt wurde. Deshalb werden auf diesen Daten keine Presence-F1- oder AUROC-Werte berichtet.

Stattdessen werden zwei Likert-Perspektiven getrennt betrachtet: Bei der **Oracle-Auswertung** wird die Item-Erkennung als perfekt angenommen. Das misst die reine Qualität des Likert-Heads. Bei der **conditional-Auswertung** werden nur Items bewertet, die der Presence-Head auswählt. Diese Auswertung ist näher an der späteren Anwendung, kann aber durch die Detection-Auswahl verzerrt sein.

## 2. Modell-Architektur

In [9]:
class BertMultiTaskModel(nn.Module):
    """Gemeinsamer Transformer-Encoder + drei Prediction-Heads."""

    def __init__(self, model_name, num_items, dropout=0.1):
        super().__init__()
        # Kommentar 1: Der Encoder wird je Experiment aus der Konfiguration geladen.
        self.encoder = AutoModel.from_pretrained(model_name, dtype=torch.float32).float()
        hidden_size = self.encoder.config.hidden_size

        # Kommentar 2: Dropout soll bei den kleinen Daten etwas gegen Overfitting helfen.
        self.dropout = nn.Dropout(dropout)
        # Kommentar 3: Jeder Head gibt pro PANAS-Item genau einen Wert zurück.
        self.presence_head = nn.Linear(hidden_size, num_items)
        self.likert_head = nn.Linear(hidden_size, num_items)
        self.cosine_head = nn.Linear(hidden_size, num_items)

    def forward(self, input_ids, attention_mask):
        # Kommentar 4: Die Attention-Maske verhindert, dass Padding den Encoder beeinflusst.
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = self.dropout(outputs.last_hidden_state[:, 0, :])

        # Kommentar 5: Die gemeinsame CLS-Repräsentation wird an alle drei Aufgaben gegeben.
        presence_logits = self.presence_head(cls_repr)
        likert_pred = self.likert_head(cls_repr)
        cosine_pred = self.cosine_head(cls_repr)
        return presence_logits, likert_pred, cosine_pred

    @torch.no_grad()
    def predict_presence_proba(self, input_ids, attention_mask):
        presence_logits, _, _ = self.forward(input_ids, attention_mask)
        return torch.sigmoid(presence_logits)

## 3. Loss-Funktionen & Maskierung

In [10]:
# Presence-Gewichte nur aus dem synthetischen Trainingssplit berechnen.
# pos_weight > 1 bestraft verpasste positive Items stärker als negative Items.
positive_counts = target_mask_train.sum(axis=0)
negative_counts = len(target_mask_train) - positive_counts
presence_pos_weight = np.divide(
    negative_counts,
    positive_counts,
    out=np.ones_like(positive_counts, dtype=np.float32),
    where=positive_counts > 0,
)
presence_pos_weight = np.clip(presence_pos_weight, 1.0, POS_WEIGHT_MAX).astype(np.float32)
presence_pos_weight_tensor = torch.tensor(presence_pos_weight, dtype=torch.float32, device=device)
presence_loss_fn = nn.BCEWithLogitsLoss(
    reduction="none", pos_weight=presence_pos_weight_tensor
)
cosine_loss_fn = nn.MSELoss()  # auf allen Samples/Items, ungemaskt

if LIKERT_LOSS_TYPE == "huber":
    likert_loss_fn = nn.SmoothL1Loss(reduction="none", beta=1.0)
else:
    likert_loss_fn = nn.MSELoss(reduction="none")

print("Presence pos_weight pro Item:")
display(pd.DataFrame({"item": PANAS_ITEMS, "positive_count": positive_counts.astype(int), "pos_weight": presence_pos_weight}).round(3))
print(f"LAMBDA_PRESENCE={LAMBDA_PRESENCE}: Presence-Loss wird im Gesamtgradienten höher gewichtet.")


def compute_losses(presence_logits, likert_pred, cosine_pred,
                   target_mask, target_values, cosine_values,
                   lambda_likert=LAMBDA_LIKERT, lambda_cosine=LAMBDA_COSINE,
                   lambda_presence=LAMBDA_PRESENCE):
    """L_total = lambda_presence * L_presence + lambda_cosine * L_cosine + lambda_likert * L_likert."""

    # Task 1: gewichtetes BCE auf der vollständigen Presence-Maske.
    loss_presence = presence_loss_fn(presence_logits, target_mask).mean()

    # Auxiliary: Cosine-Regression auf allen Items (Regularisierung).
    loss_cosine = cosine_loss_fn(cosine_pred, cosine_values)

    # Task 2: Likert -- STRENG maskiert, nur Items mit target_mask == 1.
    raw_likert_loss = likert_loss_fn(likert_pred, target_values)
    valid_items = target_mask.sum().clamp(min=1)
    loss_likert = (raw_likert_loss * target_mask).sum() / valid_items

    loss_total = (lambda_presence * loss_presence
                  + lambda_cosine * loss_cosine
                  + lambda_likert * loss_likert)
    parts = {
        "loss_presence": loss_presence.detach(),
        "loss_cosine": loss_cosine.detach(),
        "loss_likert": loss_likert.detach(),
    }
    return loss_total, parts

Presence pos_weight pro Item:


,item,positive_count,pos_weight
0,aktiv,36,20.000000
1,bekuemmert,89,9.067000
2,interessiert,50,16.920000
3,freudig_erregt,183,3.896000
4,veraergert,95,8.432000
5,stark,40,20.000000
6,schuldig,1,20.000000
7,erschrocken,6,20.000000
8,feindselig,47,18.063999
9,angeregt,74,11.108000


LAMBDA_PRESENCE=3.0: Presence-Loss wird im Gesamtgradienten höher gewichtet.


### Methodische Entscheidung: Multi-Task-Loss und Klassenungleichgewicht

Das Modell lernt Presence und Likert gemeinsam, weil beide Aufgaben inhaltlich zusammenhängen: Ein Item kann nur dann eine Likert-Intensität besitzen, wenn es im Text vorhanden ist. Der Likert-Loss wird deshalb auf gültige Items maskiert.

Die Presence-Ziele sind stark unausgeglichen. Ohne Korrektur kann das Modell fast immer die negative Klasse vorhersagen und trotzdem einen kleinen BCE-Loss erreichen. Deshalb verwendet das BCE ein per Item berechnetes `pos_weight = negative / positive`, das nur aus dem Trainingssplit stammt. Zusätzlich wird der Presence-Anteil der Gesamt-Loss mit `LAMBDA_PRESENCE = 3.0` verstärkt. Die Obergrenze `POS_WEIGHT_MAX = 20` verhindert, dass extrem seltene Items den gesamten Gradient dominieren.

Der Cosine-Loss wird nur als Auxiliary-Loss untersucht. Er soll die semantische Repräsentation unterstützen, ist aber kein direktes Gütemaß für die PANAS-Likert-Skala. Deshalb wird seine Wirkung über die Ablation mit und ohne Cosine-Loss verglichen, statt ihn als primäre Zielgröße zu interpretieren.

## 4. Training & Validierung

In [11]:
def _to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def train_epoch(model, loader, optimizer, scheduler, device,
                lambda_likert=LAMBDA_LIKERT, lambda_cosine=LAMBDA_COSINE,
                lambda_presence=LAMBDA_PRESENCE):
    """Ein Trainings-Epoch ueber den gesamten train_loader."""
    model.train()
    running = {"loss": 0.0, "loss_presence": 0.0, "loss_cosine": 0.0, "loss_likert": 0.0}

    for batch in loader:
        batch = _to_device(batch, device)

        optimizer.zero_grad()
        presence_logits, likert_pred, cosine_pred = model(batch["input_ids"], batch["attention_mask"])

        loss, parts = compute_losses(
            presence_logits, likert_pred, cosine_pred,
            batch["target_mask"], batch["target_values"], batch["cosine_values"],
            lambda_likert, lambda_cosine, lambda_presence,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running["loss"] += loss.item()
        for k, v in parts.items():
            running[k] += v.item()

    n = len(loader)
    return {k: v / n for k, v in running.items()}


@torch.no_grad()
def evaluate(model, loader, device,
             lambda_likert=LAMBDA_LIKERT, lambda_cosine=LAMBDA_COSINE,
             lambda_presence=LAMBDA_PRESENCE):
    """Loss und Metriken fuer synthetisches Testset oder externe Realdata."""
    model.eval()
    running = {"loss": 0.0, "loss_presence": 0.0, "loss_cosine": 0.0, "loss_likert": 0.0}

    all_presence_true, all_presence_prob = [], []
    all_likert_true, all_likert_pred, all_likert_mask = [], [], []
    all_cosine_true, all_cosine_pred = [], []

    for batch in loader:
        batch = _to_device(batch, device)

        presence_logits, likert_pred, cosine_pred = model(batch["input_ids"], batch["attention_mask"])
        loss, parts = compute_losses(
            presence_logits, likert_pred, cosine_pred,
            batch["target_mask"], batch["target_values"], batch["cosine_values"],
            lambda_likert, lambda_cosine, lambda_presence,
        )

        running["loss"] += loss.item()
        for k, v in parts.items():
            running[k] += v.item()

        all_presence_true.append(batch["target_mask"].cpu().numpy())
        all_presence_prob.append(torch.sigmoid(presence_logits).cpu().numpy())
        all_likert_true.append(batch["target_values"].cpu().numpy())
        all_likert_pred.append(likert_pred.cpu().numpy())
        all_likert_mask.append(batch["target_mask"].cpu().numpy())
        all_cosine_true.append(batch["cosine_values"].cpu().numpy())
        all_cosine_pred.append(cosine_pred.cpu().numpy())

    n = len(loader)
    presence_true = np.concatenate(all_presence_true, axis=0)
    presence_prob = np.concatenate(all_presence_prob, axis=0)
    presence_pred = (presence_prob > 0.5).astype(np.float32)

    f1 = f1_score(presence_true, presence_pred, average="macro", zero_division=0)
    acc = accuracy_score(presence_true.flatten(), presence_pred.flatten())
    item_aurocs = [
        roc_auc_score(presence_true[:, item_index], presence_prob[:, item_index])
        for item_index in range(NUM_ITEMS)
        if np.unique(presence_true[:, item_index]).size == 2
    ]
    auroc = float(np.mean(item_aurocs)) if item_aurocs else float("nan")

    likert_true = np.concatenate(all_likert_true, axis=0).flatten()
    likert_pred_flat = np.concatenate(all_likert_pred, axis=0).flatten()
    likert_mask_flat = np.concatenate(all_likert_mask, axis=0).flatten().astype(bool)
    if likert_mask_flat.sum() > 0:
        mae = mean_absolute_error(likert_true[likert_mask_flat], likert_pred_flat[likert_mask_flat])
        rmse = np.sqrt(mean_squared_error(likert_true[likert_mask_flat], likert_pred_flat[likert_mask_flat]))
    else:
        mae, rmse = float("nan"), float("nan")

    cosine_mae = mean_absolute_error(
        np.concatenate(all_cosine_true, axis=0).flatten(),
        np.concatenate(all_cosine_pred, axis=0).flatten(),
    )

    metrics = {k: v / n for k, v in running.items()}
    metrics["loss_main"] = metrics["loss_presence"] * lambda_presence + lambda_likert * metrics["loss_likert"]
    metrics.update({
        "task1_f1": f1,
        "task1_auroc": auroc,
        "task1_auroc_items": len(item_aurocs),
        "task1_accuracy": acc,
        "task2_mae": mae,
        "task2_rmse": rmse,
        "aux_cosine_mae": cosine_mae,
    })
    return metrics

## 5. Ablation: Training mit vs. ohne Cosine-Loss

In [12]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_model_loaders(model_name):
    """Erzeugt pro Encoder einen passenden Tokenizer und identische Loader."""
    # Kommentar 1: Jeder Encoder bekommt seinen eigenen passenden Tokenizer.
    model_tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_collate = partial(collate_fn, tokenizer=model_tokenizer, max_length=MAX_LENGTH)
    # Kommentar 2: Die Text- und Target-Splits bleiben für alle Encoder gleich.
    model_train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=model_collate)
    model_val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=model_collate)
    model_test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=model_collate)
    # Kommentar 3: Auch die externe Realdata wird nur neu tokenisiert, nicht neu bewertet.
    model_real_loader = DataLoader(real_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=model_collate)
    # Kommentar 4: So unterscheiden sich die Ansätze nur durch Encoder und Cosine-Loss.
    return model_tokenizer, model_train_loader, model_val_loader, model_test_loader, model_real_loader


def train_epoch(model, loader, optimizer, scheduler, device,
                lambda_likert=LAMBDA_LIKERT, lambda_cosine=LAMBDA_COSINE,
                lambda_presence=LAMBDA_PRESENCE):
    """Ein Trainings-Epoch ueber den gesamten train_loader."""
    model.train()
    running = {"loss": 0.0, "loss_presence": 0.0, "loss_cosine": 0.0, "loss_likert": 0.0}

    for batch in loader:
        batch = _to_device(batch, device)
        optimizer.zero_grad()
        presence_logits, likert_pred, cosine_pred = model(batch["input_ids"], batch["attention_mask"])
        loss, parts = compute_losses(
            presence_logits, likert_pred, cosine_pred,
            batch["target_mask"], batch["target_values"], batch["cosine_values"],
            lambda_likert, lambda_cosine, lambda_presence,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running["loss"] += loss.item()
        for key, value in parts.items():
            running[key] += value.item()

    n = len(loader)
    return {key: value / n for key, value in running.items()}


@torch.no_grad()
def evaluate(model, loader, device,
             lambda_likert=LAMBDA_LIKERT, lambda_cosine=LAMBDA_COSINE,
             lambda_presence=LAMBDA_PRESENCE):
    """Loss und Metriken fuer synthetisches Testset oder externe Realdata."""
    model.eval()
    running = {"loss": 0.0, "loss_presence": 0.0, "loss_cosine": 0.0, "loss_likert": 0.0}
    all_presence_true, all_presence_prob = [], []
    all_likert_true, all_likert_pred, all_likert_mask = [], [], []
    all_cosine_true, all_cosine_pred = [], []

    for batch in loader:
        batch = _to_device(batch, device)
        presence_logits, likert_pred, cosine_pred = model(batch["input_ids"], batch["attention_mask"])
        loss, parts = compute_losses(
            presence_logits, likert_pred, cosine_pred,
            batch["target_mask"], batch["target_values"], batch["cosine_values"],
            lambda_likert, lambda_cosine, lambda_presence,
        )
        running["loss"] += loss.item()
        for key, value in parts.items():
            running[key] += value.item()

        all_presence_true.append(batch["target_mask"].cpu().numpy())
        all_presence_prob.append(torch.sigmoid(presence_logits).cpu().numpy())
        all_likert_true.append(batch["target_values"].cpu().numpy())
        all_likert_pred.append(likert_pred.cpu().numpy())
        all_likert_mask.append(batch["target_mask"].cpu().numpy())
        all_cosine_true.append(batch["cosine_values"].cpu().numpy())
        all_cosine_pred.append(cosine_pred.cpu().numpy())

    n = len(loader)
    presence_true = np.concatenate(all_presence_true, axis=0)
    presence_prob = np.concatenate(all_presence_prob, axis=0)
    presence_pred = (presence_prob > 0.5).astype(np.float32)
    f1 = f1_score(presence_true, presence_pred, average="macro", zero_division=0)
    acc = accuracy_score(presence_true.flatten(), presence_pred.flatten())
    item_aurocs = [
        roc_auc_score(presence_true[:, item_index], presence_prob[:, item_index])
        for item_index in range(NUM_ITEMS)
        if np.unique(presence_true[:, item_index]).size == 2
    ]
    auroc = float(np.mean(item_aurocs)) if item_aurocs else float("nan")

    likert_true = np.concatenate(all_likert_true, axis=0).flatten()
    likert_pred_flat = np.concatenate(all_likert_pred, axis=0).flatten()
    likert_mask_flat = np.concatenate(all_likert_mask, axis=0).flatten().astype(bool)
    if likert_mask_flat.sum() > 0:
        mae = mean_absolute_error(likert_true[likert_mask_flat], likert_pred_flat[likert_mask_flat])
        rmse = np.sqrt(mean_squared_error(likert_true[likert_mask_flat], likert_pred_flat[likert_mask_flat]))
    else:
        mae, rmse = float("nan"), float("nan")

    cosine_mae = mean_absolute_error(
        np.concatenate(all_cosine_true, axis=0).flatten(),
        np.concatenate(all_cosine_pred, axis=0).flatten(),
    )
    metrics = {key: value / n for key, value in running.items()}
    metrics["loss_main"] = metrics["loss_presence"] * lambda_presence + lambda_likert * metrics["loss_likert"]
    metrics.update({
        "task1_f1": f1,
        "task1_auroc": auroc,
        "task1_auroc_items": len(item_aurocs),
        "task1_accuracy": acc,
        "task2_mae": mae,
        "task2_rmse": rmse,
        "aux_cosine_mae": cosine_mae,
    })
    return metrics


def run_experiment(name, encoder_key, lambda_cosine, seed, epochs=EPOCHS):
    """Trainiert einen Encoder und waehlt den Checkpoint auf der Validation."""
    set_seed(seed)
    model_name = ENCODER_CONFIGS[encoder_key]["model_name"]
    # Kommentar 5: Vor jedem Lauf werden Seed, Modell und Optimizer neu gesetzt.
    model_tokenizer, model_train_loader, model_val_loader, model_test_loader, model_real_loader = build_model_loaders(model_name)
    model = BertMultiTaskModel(model_name, num_items=NUM_ITEMS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(model_train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )

    ckpt_path = f"best_{name}_seed{seed}.pt"
    best_main_loss, best, best_test, history = float("inf"), None, None, []
    for epoch in range(1, epochs + 1):
        train_metrics = train_epoch(
            model, model_train_loader, optimizer, scheduler, device,
            LAMBDA_LIKERT, lambda_cosine, LAMBDA_PRESENCE,
        )
        val_metrics = evaluate(
            model, model_val_loader, device,
            LAMBDA_LIKERT, lambda_cosine, LAMBDA_PRESENCE,
        )
        test_metrics = evaluate(
            model, model_test_loader, device,
            LAMBDA_LIKERT, lambda_cosine, LAMBDA_PRESENCE,
        )
        history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics, "test": test_metrics})
        print(
            f"[{name} | seed {seed}] Epoch {epoch}/{epochs} | "
            f"Train={train_metrics['loss']:.4f} | Val main={val_metrics['loss_main']:.4f} "
            f"F1={val_metrics['task1_f1']:.3f} MAE={val_metrics['task2_mae']:.3f} | "
            f"Test F1={test_metrics['task1_f1']:.3f} MAE={test_metrics['task2_mae']:.3f}"
        )
        if val_metrics["loss_main"] < best_main_loss:
            best_main_loss = val_metrics["loss_main"]
            best = {**val_metrics, "epoch": epoch}
            best_test = {**test_metrics, "epoch": epoch}
            torch.save(model.state_dict(), ckpt_path)

    del model, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "name": name,
        "encoder": encoder_key,
        "model_name": model_name,
        "seed": seed,
        "lambda_cosine": lambda_cosine,
        "ckpt_path": ckpt_path,
        "tokenizer": model_tokenizer,
        "real_loader": model_real_loader,
        "best": best,
        "best_test": best_test,
        "history": history,
    }


all_runs = []
for seed in SEEDS:
    for name, experiment in EXPERIMENTS.items():
        print("=" * 100)
        print(f"Experiment: {name} | seed={seed}")
        print("=" * 100)
        all_runs.append(run_experiment(name, experiment["encoder"], experiment["lambda_cosine"], seed))

Experiment: bert_ohne_cosine | seed=42


c:\Users\StrasserNora\Desktop\NorasNotizen\privat\NLE\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\StrasserNora\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\StrasserNora\Desktop\NorasNotizen\privat\NLE\.venv\Lib\site-packages\huggingface_hub\file_download.py:

KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------------------
# Externe Realdata-Validierung: nur Likert auf Presence-erkannten Items
# ---------------------------------------------------------------------------
def evaluate_real_likert(model, loader, device, presence_threshold=REAL_PRESENCE_THRESHOLD):
    model.eval()
    all_true, all_pred, all_detected, all_valid = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = _to_device(batch, device)
            presence_logits, likert_pred, _ = model(batch["input_ids"], batch["attention_mask"])
            detected = torch.sigmoid(presence_logits) >= presence_threshold
            all_true.append(batch["target_values"].cpu().numpy())
            all_pred.append(likert_pred.cpu().numpy())
            all_detected.append(detected.cpu().numpy())
            all_valid.append(batch["target_mask"].cpu().numpy().astype(bool))

    y_true = np.concatenate(all_true, axis=0)
    y_pred = np.clip(np.concatenate(all_pred, axis=0), 1.0, 5.0)
    detected = np.concatenate(all_detected, axis=0).astype(bool)
    valid = np.concatenate(all_valid, axis=0).astype(bool)
    selected = detected & valid

    true_selected = y_true[selected]
    pred_selected = y_pred[selected]
    rounded_pred = np.clip(np.floor(pred_selected + 0.5), 1, 5).astype(int)
    rounded_true = true_selected.astype(int)
    if len(true_selected):
        overall = {
            "valid_ratings": int(valid.sum()),
            "evaluated_ratings": int(selected.sum()),
            "coverage": float(selected.sum() / max(valid.sum(), 1)),
            "likert_mae": float(mean_absolute_error(true_selected, pred_selected)),
            "likert_rmse": float(np.sqrt(mean_squared_error(true_selected, pred_selected))),
            "rounded_accuracy": float(accuracy_score(rounded_true, rounded_pred)),
        }
    else:
        overall = {"valid_ratings": int(valid.sum()), "evaluated_ratings": 0, "coverage": 0.0,
                   "likert_mae": np.nan, "likert_rmse": np.nan, "rounded_accuracy": np.nan}

    item_rows = []
    item_qwks = []
    for item_index, item in enumerate(REAL_PANAS_COLUMNS):
        item_valid = valid[:, item_index]
        item_selected = selected[:, item_index]
        item_true = y_true[item_selected, item_index].astype(int)
        item_pred = np.clip(np.floor(y_pred[item_selected, item_index] + 0.5), 1, 5).astype(int)
        qwk = np.nan
        if len(item_true) and np.unique(item_true).size > 1:
            qwk = cohen_kappa_score(item_true, item_pred, weights="quadratic")
            if np.isfinite(qwk):
                item_qwks.append(qwk)
        item_rows.append({
            "item": item,
            "valid_ratings": int(item_valid.sum()),
            "detected_ratings": int(item_selected.sum()),
            "coverage": float(item_selected.sum() / max(item_valid.sum(), 1)),
            "MAE": float(mean_absolute_error(y_true[item_selected, item_index], y_pred[item_selected, item_index])) if item_selected.any() else np.nan,
            "RMSE": float(np.sqrt(mean_squared_error(y_true[item_selected, item_index], y_pred[item_selected, item_index])) if item_selected.any() else np.nan),
            "rounded_accuracy": float(accuracy_score(item_true, item_pred)) if len(item_true) else np.nan,
            "QWK": float(qwk) if np.isfinite(qwk) else np.nan,
        })
    overall["mean_item_qwk"] = float(np.mean(item_qwks)) if item_qwks else np.nan
    return overall, pd.DataFrame(item_rows)


real_validation_rows = []
real_item_tables = []
for run in all_runs:
    # Fuenf Kommentare in diesem Feld erklaeren Input, Maskierung und Vergleich.
    validation_model = BertMultiTaskModel(run["model_name"], num_items=NUM_ITEMS).to(device)
    validation_model.load_state_dict(torch.load(run["ckpt_path"], map_location=device))
    metrics, item_metrics = evaluate_real_likert(validation_model, run["real_loader"], device)
    metrics.update({
        "experiment": run["name"],
        "encoder": run["encoder"],
        "seed": run["seed"],
        "best_epoch_synthetic": run["best"]["epoch"],
    })
    real_validation_rows.append(metrics)
    item_metrics.insert(0, "seed", run["seed"])
    item_metrics.insert(0, "encoder", run["encoder"])
    item_metrics.insert(0, "experiment", run["name"])
    real_item_tables.append(item_metrics)
    del validation_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

real_validation_results = pd.DataFrame(real_validation_rows)
real_item_results = pd.concat(real_item_tables, ignore_index=True)
print("Externe Validierung auf real_data.csv; Presence wird nicht bewertet.")
print(f"Likert-Auswertung nur bei Presence-Score >= {REAL_PRESENCE_THRESHOLD:.2f}.")
display(real_validation_results[["experiment", "encoder", "seed", "best_epoch_synthetic", "valid_ratings", "evaluated_ratings", "coverage", "likert_mae", "likert_rmse", "rounded_accuracy", "mean_item_qwk"]].round(4))
print("Itemweise externe Likert-Metriken:")
display(real_item_results.round(3))

summary = real_validation_results.groupby("experiment")[["likert_mae", "likert_rmse", "rounded_accuracy", "coverage"]].mean()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
summary["likert_mae"].plot.bar(ax=axes[0], color="#3274A1", title="Likert MAE ↓")
summary["rounded_accuracy"].plot.bar(ax=axes[1], color="#3A923A", title="Gerundete Accuracy ↑")
summary["coverage"].plot.bar(ax=axes[2], color="#E1812C", title="Presence-Coverage ↑")
for ax in axes:
    ax.set_xlabel("Experiment")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=35)
plt.suptitle("Externe Validierung auf echten PANAS-Likertdaten")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# ERWEITERTE EXTERNE VALIDIERUNG
# ---------------------------------------------------------------------------
# Oracle = alle gueltigen Ratings, also perfekte Item-Erkennung.
# Conditional = nur Ratings oberhalb des jeweiligen Presence-Thresholds.
# Mehrere Thresholds zeigen den Zielkonflikt zwischen Coverage und Fehler.
BOOTSTRAP_ITERATIONS = 500
BOOTSTRAP_RANDOM_STATE = 123


def _likert_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "exact_accuracy": np.nan,
                "within_1_accuracy": np.nan, "QWK": np.nan, "bias": np.nan}
    rounded_pred = np.clip(np.floor(y_pred + 0.5), 1, 5).astype(int)
    qwk = np.nan
    if np.unique(y_true).size > 1:
        qwk = cohen_kappa_score(y_true, rounded_pred, weights="quadratic")
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "exact_accuracy": float(accuracy_score(y_true, rounded_pred)),
        "within_1_accuracy": float(np.mean(np.abs(y_true - rounded_pred) <= 1)),
        "QWK": float(qwk) if np.isfinite(qwk) else np.nan,
        "bias": float(np.mean(y_pred - y_true)),
    }


def _group_bootstrap_ci(y_true, y_pred, selected, groups, iterations=BOOTSTRAP_ITERATIONS):
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
    values = []
    for _ in range(iterations):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        sampled_rows = np.concatenate([np.flatnonzero(groups == group) for group in sampled_groups])
        sampled_selected = selected[sampled_rows]
        if not sampled_selected.any():
            continue
        metrics = _likert_metrics(y_true[sampled_rows][sampled_selected], y_pred[sampled_rows][sampled_selected])
        values.append([metrics["MAE"], metrics["QWK"]])
    if not values:
        return {"MAE_ci_low": np.nan, "MAE_ci_high": np.nan, "QWK_ci_low": np.nan, "QWK_ci_high": np.nan}
    values = np.asarray(values, dtype=float)
    return {
        "MAE_ci_low": float(np.nanpercentile(values[:, 0], 2.5)),
        "MAE_ci_high": float(np.nanpercentile(values[:, 0], 97.5)),
        "QWK_ci_low": float(np.nanpercentile(values[:, 1], 2.5)),
        "QWK_ci_high": float(np.nanpercentile(values[:, 1], 97.5)),
    }


def _predict_real_arrays(model, loader):
    model.eval()
    all_pred, all_presence = [], []
    with torch.no_grad():
        for batch in loader:
            batch = _to_device(batch, device)
            presence_logits, likert_pred, _ = model(batch["input_ids"], batch["attention_mask"])
            all_presence.append(torch.sigmoid(presence_logits).cpu().numpy())
            all_pred.append(likert_pred.cpu().numpy())
    return np.clip(np.concatenate(all_pred, axis=0), 1.0, 5.0), np.concatenate(all_presence, axis=0)


train_valid = target_mask_train.astype(bool)
synthetic_mean_baseline = np.divide(
    (target_values_train * train_valid).sum(axis=0),
    train_valid.sum(axis=0),
    out=np.full(NUM_ITEMS, 3.0, dtype=float),
    where=train_valid.sum(axis=0) > 0,
)
synthetic_mode_baseline = np.array([
    pd.Series(target_values_train[train_valid[:, item_index], item_index]).mode().iloc[0]
    if train_valid[:, item_index].any() else 3.0
    for item_index in range(NUM_ITEMS)
])

extended_rows = []
bootstrap_rows = []
for run in all_runs:
    # Kommentar 1: Der passende Encoder-Checkpoint wird für jeden Lauf geladen.
    validation_model = BertMultiTaskModel(run["model_name"], num_items=NUM_ITEMS).to(device)
    validation_model.load_state_dict(torch.load(run["ckpt_path"], map_location=device))
    # Kommentar 2: Die Realdata wird mit dem zum Encoder gehörenden Loader verarbeitet.
    y_pred, presence_prob = _predict_real_arrays(validation_model, run["real_loader"])
    y_true = real_target_values
    valid = real_target_mask.astype(bool)

    # Kommentar 3: Oracle trennt Likert-Qualität von der nicht validierbaren Detection.
    oracle_metrics = _likert_metrics(y_true[valid], y_pred[valid])
    oracle_baseline_mean = _likert_metrics(y_true[valid], np.broadcast_to(synthetic_mean_baseline, y_true.shape)[valid])
    oracle_baseline_mode = _likert_metrics(y_true[valid], np.broadcast_to(synthetic_mode_baseline, y_true.shape)[valid])
    extended_rows.extend([
        {"experiment": run["name"], "encoder": run["encoder"], "seed": run["seed"], "evaluation": "oracle_model", **oracle_metrics,
         "coverage": 1.0, "evaluated_ratings": int(valid.sum())},
        {"experiment": run["name"], "encoder": run["encoder"], "seed": run["seed"], "evaluation": "oracle_baseline_mean", **oracle_baseline_mean,
         "coverage": 1.0, "evaluated_ratings": int(valid.sum())},
        {"experiment": run["name"], "encoder": run["encoder"], "seed": run["seed"], "evaluation": "oracle_baseline_mode", **oracle_baseline_mode,
         "coverage": 1.0, "evaluated_ratings": int(valid.sum())},
    ])

    for threshold in REAL_PRESENCE_THRESHOLDS:
        # Kommentar 4: Conditional bewertet nur erkannte gültige Ratings.
        selected = (presence_prob >= threshold) & valid
        metrics = _likert_metrics(y_true[selected], y_pred[selected])
        row = {
            "experiment": run["name"], "encoder": run["encoder"], "seed": run["seed"],
            "evaluation": "conditional_model", "threshold": threshold,
            "coverage": float(selected.sum() / max(valid.sum(), 1)),
            "evaluated_ratings": int(selected.sum()), **metrics,
        }
        extended_rows.append(row)
        # Kommentar 5: Bootstrap zieht ganze Personen-Gruppen statt einzelne Ratings.
        ci = _group_bootstrap_ci(y_true, y_pred, selected, real_groups)
        bootstrap_rows.append({**row, **ci})

    del validation_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

extended_validation_results = pd.DataFrame(extended_rows)
conditional_threshold_results = extended_validation_results[
    extended_validation_results["evaluation"] == "conditional_model"
].copy()
bootstrap_results = pd.DataFrame(bootstrap_rows)

print("\n" + "=" * 90)
print("ERWEITERTE REALDATA-VALIDIERUNG")
print("Oracle = Detection wird als perfekt angenommen; misst reine Likert-Leistung.")
print("Conditional = nur erkannte Items; misst die reale Pipeline, aber selektiv.")
print("Baselines = synthetischer Train-Mittelwert bzw. synthetischer Train-Modus.")
print("Konfidenzintervalle werden nach name/Person gruppiert gebootstrapped.")
print("=" * 90)

oracle_summary = extended_validation_results[
    extended_validation_results["evaluation"].str.startswith("oracle_")
].groupby("evaluation")[["MAE", "RMSE", "exact_accuracy", "within_1_accuracy", "QWK", "bias"]].mean()
display(oracle_summary.round(4))
threshold_summary = conditional_threshold_results.groupby(["experiment", "encoder", "threshold"])[
    ["MAE", "RMSE", "exact_accuracy", "within_1_accuracy", "QWK", "bias", "coverage"]
].mean()
print("Conditional-Ergebnisse über mehrere Presence-Thresholds:")
display(threshold_summary.round(4))
print("Gruppierte 95%-Bootstrap-Konfidenzintervalle für MAE und QWK:")
display(bootstrap_results[["experiment", "encoder", "seed", "threshold", "coverage", "MAE", "MAE_ci_low", "MAE_ci_high", "QWK", "QWK_ci_low", "QWK_ci_high"]].round(4))

## 6. Zusätzliche Realdata-Plausibilitätsanalyse und Modellvergleich

In [14]:
# ---------------------------------------------------------------------------
# Zusätzliche Realdata-Analyse: Personenmittelwerte und manuelle Plausibilität
# ---------------------------------------------------------------------------
# Kommentar 1: Die ersten zehn PANAS-Items werden als positiv, die letzten zehn als negativ behandelt.
POSITIVE_ITEM_INDICES = np.arange(0, NUM_ITEMS // 2)
NEGATIVE_ITEM_INDICES = np.arange(NUM_ITEMS // 2, NUM_ITEMS)
POSITIVE_ITEM_NAMES = [REAL_PANAS_COLUMNS[index] for index in POSITIVE_ITEM_INDICES]
NEGATIVE_ITEM_NAMES = [REAL_PANAS_COLUMNS[index] for index in NEGATIVE_ITEM_INDICES]

# Kommentar 2: Die Realdata wird einmal pro vorhandenem Checkpoint vorhergesagt.
realdata_person_rows = []
realdata_prediction_tables = []
realdata_arrays = {}

for run in all_runs:
    validation_model = BertMultiTaskModel(run["model_name"], num_items=NUM_ITEMS).to(device)
    validation_model.load_state_dict(torch.load(run["ckpt_path"], map_location=device))
    y_pred, presence_prob = _predict_real_arrays(validation_model, run["real_loader"])
    y_pred = np.clip(y_pred, 1.0, 5.0)
    detected = presence_prob >= REAL_PRESENCE_THRESHOLD
    gated_pred = np.where(detected, y_pred, 0.0)
    realdata_arrays[(run["name"], run["seed"])] = {
        "y": real_target_values.copy(),
        "pred": y_pred,
        "presence_prob": presence_prob,
        "gated_pred": gated_pred,
    }

    # Kommentar 3: Für jede Aufnahme werden die gültigen positiven und negativen Items gemittelt.
    valid = real_target_mask.astype(bool)
    valid_pos_count = valid[:, POSITIVE_ITEM_INDICES].sum(axis=1)
    valid_neg_count = valid[:, NEGATIVE_ITEM_INDICES].sum(axis=1)
    y_pos = np.divide(
        np.where(valid[:, POSITIVE_ITEM_INDICES], real_target_values[:, POSITIVE_ITEM_INDICES], 0.0).sum(axis=1),
        valid_pos_count,
        out=np.full(len(real_target_values), np.nan),
        where=valid_pos_count > 0,
    )
    y_neg = np.divide(
        np.where(valid[:, NEGATIVE_ITEM_INDICES], real_target_values[:, NEGATIVE_ITEM_INDICES], 0.0).sum(axis=1),
        valid_neg_count,
        out=np.full(len(real_target_values), np.nan),
        where=valid_neg_count > 0,
    )
    pred_pos = y_pred[:, POSITIVE_ITEM_INDICES].mean(axis=1)
    pred_neg = y_pred[:, NEGATIVE_ITEM_INDICES].mean(axis=1)

    row_table = pd.DataFrame({
        "name": real_groups,
        "y_pos_mean": y_pos,
        "y_neg_mean": y_neg,
        "pred_pos_mean": pred_pos,
        "pred_neg_mean": pred_neg,
    })
    # Kommentar 4: Wiederholte Aufnahmen derselben Person werden zu Personenmitteln zusammengefasst.
    person_table = row_table.groupby("name", as_index=False).mean(numeric_only=True)
    person_table.insert(0, "experiment", run["name"])
    person_table.insert(1, "encoder", run["encoder"])
    person_table.insert(2, "seed", run["seed"])
    realdata_person_rows.append(person_table)

    # Kommentar 5: Die Tabelle enthält bewusst alle 20 Vorhersagen; nicht ausgewählte Items werden 0.
    text_table = pd.DataFrame({"experiment": run["name"], "encoder": run["encoder"], "seed": run["seed"], "name": real_groups, "text": real_text})
    for item_index, item in enumerate(REAL_PANAS_COLUMNS):
        text_table[f"pred_{item}"] = gated_pred[:, item_index]
    realdata_prediction_tables.append(text_table)

    del validation_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

realdata_person_means = pd.concat(realdata_person_rows, ignore_index=True) if realdata_person_rows else pd.DataFrame()
realdata_prediction_table = pd.concat(realdata_prediction_tables, ignore_index=True) if realdata_prediction_tables else pd.DataFrame()

print("Personenmittelwerte aus Realdata und Modellvorhersagen:")
display(realdata_person_means.round(3))
print("Alle Realdata-Texte mit 20 Presence-gegateten Likert-Vorhersagen (0 = nicht ausgewählt):")
display(realdata_prediction_table.round(3))
if not all_runs:
    print("Noch keine vollständigen Checkpoints vorhanden; die Tabellen werden nach dem Training automatisch gefüllt.")

Personenmittelwerte aus Realdata und Modellvorhersagen:


""


Alle Realdata-Texte mit 20 Presence-gegateten Likert-Vorhersagen (0 = nicht ausgewählt):


""


Noch keine vollständigen Checkpoints vorhanden; die Tabellen werden nach dem Training automatisch gefüllt.


In [15]:
# ---------------------------------------------------------------------------
# Korrelationen der positiven/negativen Mittelwerte und 8-Balken-Vergleich
# ---------------------------------------------------------------------------
# Kommentar 1: Hier werden die echten Realdata-Werte und Modellwerte getrennt verglichen.
correlation_rows = []
for (experiment_name, seed), arrays in realdata_arrays.items():
    person_subset = realdata_person_means[
        (realdata_person_means["experiment"] == experiment_name)
        & (realdata_person_means["seed"] == seed)
    ].copy()
    true_pos_neg = person_subset["y_pos_mean"].corr(person_subset["y_neg_mean"], method="pearson")
    pred_pos_neg = person_subset["pred_pos_mean"].corr(person_subset["pred_neg_mean"], method="pearson")
    true_pred_pos = person_subset["y_pos_mean"].corr(person_subset["pred_pos_mean"], method="pearson")
    true_pred_neg = person_subset["y_neg_mean"].corr(person_subset["pred_neg_mean"], method="pearson")
    true_pos_neg_spearman = person_subset["y_pos_mean"].corr(person_subset["y_neg_mean"], method="spearman")
    pred_pos_neg_spearman = person_subset["pred_pos_mean"].corr(person_subset["pred_neg_mean"], method="spearman")
    correlation_rows.append({
        "experiment": experiment_name,
        "seed": seed,
        "true_pos_neg_pearson": true_pos_neg,
        "pred_pos_neg_pearson": pred_pos_neg,
        "true_pos_neg_spearman": true_pos_neg_spearman,
        "pred_pos_neg_spearman": pred_pos_neg_spearman,
        "true_pred_pos_pearson": true_pred_pos,
        "true_pred_neg_pearson": true_pred_neg,
    })

# Kommentar 2: Die Korrelation true_pos_neg gegen pred_pos_neg beantwortet die gewünschte Ähnlichkeitsfrage.
correlation_results = pd.DataFrame(correlation_rows)
print("Korrelation der Personenmittelwerte von positiven und negativen Items:")
if correlation_results.empty:
    print("Noch keine vollständigen Modellläufe vorhanden; Korrelationen werden nach dem Training berechnet.")
else:
    display(correlation_results.round(3))

# Kommentar 3: Für den Barplot werden ausschließlich synthetische Testlabels verwendet.
plot_rows = []
for run in all_runs:
    if run.get("best_test") is None:
        continue
    plot_rows.append({
        "experiment": run["name"],
        "encoder": ENCODER_CONFIGS[run["encoder"]]["label"],
        "seed": run["seed"],
        "accuracy": run["best_test"]["task1_accuracy"],
        "f1": run["best_test"]["task1_f1"],
    })

# Kommentar 4: Mittelwerte über Seeds machen die vier Ansätze direkt vergleichbar.
plot_results = pd.DataFrame(plot_rows)
if plot_results.empty:
    print("Noch keine vollständigen Testmetriken vorhanden; der Barplot wird nach dem Training erzeugt.")
else:
    plot_means = plot_results.groupby(["experiment", "encoder"])[["accuracy", "f1"]].mean().reset_index()
    x = np.arange(len(plot_means))
    width = 0.36
    fig, ax = plt.subplots(figsize=(12, 5))
    accuracy_bars = ax.bar(x - width / 2, plot_means["accuracy"], width, label="Accuracy", color="#3B82F6")
    f1_bars = ax.bar(x + width / 2, plot_means["f1"], width, label="Macro-F1", color="#F97316")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_means["experiment"], rotation=20, ha="right")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.set_title("Presence-Vergleich auf dem synthetischen Testset")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    for bars in (accuracy_bars, f1_bars):
        ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.show()

# Kommentar 5: Das Diagramm enthält vier Ansätze mit jeweils Accuracy und Macro-F1, also acht Balken.
if not plot_results.empty:
    display(plot_means.round(4))

Korrelation der Personenmittelwerte von positiven und negativen Items:
Noch keine vollständigen Modellläufe vorhanden; Korrelationen werden nach dem Training berechnet.
Noch keine vollständigen Testmetriken vorhanden; der Barplot wird nach dem Training erzeugt.


### Methodische Entscheidung: Baselines, Thresholds und Konfidenzintervalle

Die Baselines zeigen, ob das Modell die echten Daten besser vorhersagt als eine einfache Vorhersage aus dem synthetischen Training. Ohne diese Referenzen wäre ein MAE allein schwer einzuordnen.

Mehrere Presence-Thresholds machen den Zielkonflikt sichtbar: Ein niedriger Threshold erhöht meist die Coverage, kann aber mehr ungeeignete Likert-Vorhersagen einschließen. Ein hoher Threshold kann die conditional-Metriken verbessern, bewertet dann aber möglicherweise nur noch wenige Ratings. Daher sollte die Schwelle nicht ausschließlich nach MAE ausgewählt werden.

Die Ratings sind innerhalb einer Person nicht unabhängig. Deshalb werden die Bootstrap-Konfidenzintervalle nach `name` gruppiert. So werden ganze Personen-Gruppen gezogen und die Unsicherheit nicht durch wiederholte Aufnahmen derselben Person künstlich unterschätzt.

In [ ]:
# ---------------------------------------------------------------------------
# Hauptvergleich auf Realdata: nur externe Likert-Validierung
# ---------------------------------------------------------------------------
# Die erweiterte Auswertung steht in der vorherigen Zelle. Diese Tabelle bleibt
# als kompakter Vergleich der bisherigen bedingten Metriken erhalten.
REAL_COMPARISON_METRICS = {
    "likert_mae": "Likert MAE (↓)",
    "likert_rmse": "Likert RMSE (↓)",
    "rounded_accuracy": "Gerundete Accuracy (↑)",
    "coverage": "Presence-Coverage (↑)",
    "mean_item_qwk": "Mittleres Item-QWK (↑)",
}

real_summary = real_validation_results.groupby("experiment")[list(REAL_COMPARISON_METRICS)].agg(["mean", "std"])
real_table = real_summary.xs("mean", axis=1, level=1).rename(columns=REAL_COMPARISON_METRICS)
print("Hauptvergleich auf echten Daten (Mittelwert ueber Seeds):")
display(
    real_table.style
    .format("{:.4f}")
    .highlight_min(subset=["Likert MAE (↓)", "Likert RMSE (↓)"], color="#d9ead3")
    .highlight_max(subset=["Gerundete Accuracy (↑)", "Presence-Coverage (↑)", "Mittleres Item-QWK (↑)"], color="#d9ead3")
    .set_caption("Bisherige bedingte Realdata-Validierung")
)

if {"ohne_cosine", "mit_cosine"} <= set(real_table.index):
    real_direct = pd.DataFrame({
        "ohne_cosine": real_table.loc["ohne_cosine"],
        "mit_cosine": real_table.loc["mit_cosine"],
    })
    real_direct["diff (mit - ohne)"] = real_direct["mit_cosine"] - real_direct["ohne_cosine"]
    real_direct["besser mit Cosine?"] = [
        (diff < 0) if metric in {"Likert MAE (↓)", "Likert RMSE (↓)"} else (diff > 0)
        for metric, diff in zip(real_direct.index, real_direct["diff (mit - ohne)"])
    ]
    print("Direkter Realdata-Vergleich:")
    display(real_direct.style.format({"ohne_cosine": "{:.4f}", "mit_cosine": "{:.4f}", "diff (mit - ohne)": "{:+.4f}"}))

plot_data = real_table.reset_index().melt(id_vars="experiment", var_name="metric", value_name="value")
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes.flat, ["Likert MAE (↓)", "Likert RMSE (↓)", "Gerundete Accuracy (↑)", "Presence-Coverage (↑)"]):
    values = plot_data[plot_data["metric"] == metric]
    bars = ax.bar(values["experiment"], values["value"], color=["#355c7d", "#c06c84"])
    ax.set_title(metric)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=20)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{bar.get_height():.3f}",
                ha="center", va="bottom", fontsize=9)
fig.suptitle("Bedingter Modellvergleich auf echten PANAS-Likertdaten", fontsize=14)
fig.tight_layout()
plt.show()

## 6. Vergleich: mit vs. ohne Cosine-Loss

Die synthetische Testauswertung dient nur zur Checkpoint-Auswahl. Der fachlich relevante Modellvergleich erfolgt anschliessend auf der externen Realdata-Validierung: bewertet werden ausschliesslich Likert-Werte fuer Items, die das Modell per Presence-Score erkannt hat. Presence selbst wird auf den Realdata nicht als Ground Truth bewertet.

In [ ]:
# Synthetische Testmetriken am jeweils auf Validation ausgewählten Checkpoint.
METRIC_COLS = ["loss_main", "task1_f1", "task1_auroc", "task1_accuracy", "task2_mae", "task2_rmse"]
HIGHER_IS_BETTER = {"task1_f1", "task1_auroc", "task1_accuracy"}

synthetic_test_results = pd.DataFrame([
    {"experiment": r["name"], "seed": r["seed"], "best_epoch": r["best_test"]["epoch"],
     **{m: r["best_test"][m] for m in METRIC_COLS}}
    for r in all_runs
])
print("Synthetischer Testvergleich pro Lauf (Checkpoint auf Validation gewählt):")
display(synthetic_test_results.round(4))

synthetic_means = synthetic_test_results.groupby("experiment")[METRIC_COLS].mean()
print("\nMittelwert auf dem synthetischen Testset:")
display(synthetic_means.round(4))

if {"ohne_cosine", "mit_cosine"} <= set(synthetic_means.index):
    synthetic_comparison = pd.DataFrame({
        "ohne_cosine": synthetic_means.loc["ohne_cosine"],
        "mit_cosine": synthetic_means.loc["mit_cosine"],
    })
    synthetic_comparison["diff (mit - ohne)"] = synthetic_comparison["mit_cosine"] - synthetic_comparison["ohne_cosine"]
    synthetic_comparison["besser mit Cosine?"] = [
        (diff > 0) if metric in HIGHER_IS_BETTER else (diff < 0)
        for metric, diff in zip(synthetic_comparison.index, synthetic_comparison["diff (mit - ohne)"])
    ]
    print("\nDirekter synthetischer Testvergleich:")
    display(synthetic_comparison.round(4))

In [ ]:
# Synthetische Testkurven pro Epoche (Mittelwert ueber Seeds, Band = Std.)
plot_metrics = [("loss_main", "Synth-Test Loss (Presence + Likert)"),
                ("task1_f1", "Synth-Test F1 (Presence)"),
                ("task2_mae", "Synth-Test MAE (Likert)")]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(15, 4))
for ax, (key, title) in zip(axes, plot_metrics):
    for name in EXPERIMENTS:
        curves = np.array([[h["test"][key] for h in r["history"]] for r in all_runs if r["name"] == name])
        mean, x = curves.mean(axis=0), np.arange(1, curves.shape[1] + 1)
        ax.plot(x, mean, marker="o", label=name)
        if len(curves) > 1:
            ax.fill_between(x, mean - curves.std(axis=0), mean + curves.std(axis=0), alpha=0.2)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Bestes Modell laden (optional)

Lädt den besten Checkpoint der unten gewählten Variante (bester Seed nach `loss_main`).
Für einen Domain-Shift-Check auf `real_data.csv` kann mit diesem Modell `evaluate()` auf einem
entsprechend gebauten `DataLoader` laufen.

In [ ]:
FINAL_EXPERIMENT = "mit_cosine"   # oder "ohne_cosine"

final_run = min((r for r in all_runs if r["name"] == FINAL_EXPERIMENT), key=lambda r: r["best"]["loss_main"])
best_model = BertMultiTaskModel(MODEL_NAME, num_items=NUM_ITEMS).to(device)
best_model.load_state_dict(torch.load(final_run["ckpt_path"], map_location=device))
best_model.eval()
print(f"Geladen: {final_run['ckpt_path']} (Epoch {final_run['best']['epoch']})")


### Grenzen und geplante Sensitivitätsanalyse

Die externe Stichprobe ist relativ klein und enthält wiederholte Aufnahmen pro Person. Die gruppierten Konfidenzintervalle berücksichtigen diese Abhängigkeit, ersetzen aber keine größere unabhängige Stichprobe.

Außerdem werden derzeit 11 Transkripte bei `MAX_LENGTH = 256` abgeschnitten. Deshalb wird im Output ausdrücklich eine Sensitivitätsanalyse mit `MAX_LENGTH = 512` empfohlen. Die Ergebnisse sollten nur dann als robust bezeichnet werden, wenn sich die zentralen Oracle- und conditional-Metriken dadurch nicht wesentlich verändern.

Für die Interpretation der Hausarbeit sind daher vor allem drei Fragen entscheidend: Schlägt das Modell beide Baselines? Bleibt die Oracle-Likert-Leistung über mehrere Seeds stabil? Und bietet ein Threshold einen vertretbaren Kompromiss zwischen Fehler und Coverage?